# Kamerplanter RAG Quality Benchmark

Interaktive Auswertung der RAG-Pipeline-Qualitaet (REQ-031).

**Voraussetzungen** (via Skaffold port-forwards):
- Embedding Service auf `localhost:8080`
- VectorDB (pgvector) auf `localhost:5433`
- Ollama auf `localhost:11434` mit geladenem Modell

**Ablauf:**
1. Konfiguration & Verbindungstest
2. Benchmark ausfuehren (100 Fragen durch RAG-Pipeline)
3. Ergebnisse visualisieren
4. Fehleranalyse

## 0. Setup

In [ ]:
%pip install -q httpx psycopg[binary] pyyaml matplotlib pandas

In [ ]:
import json
import re
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path

import httpx
import matplotlib.pyplot as plt
import pandas as pd
import psycopg
import yaml

## 1. Konfiguration

In [ ]:
# Endpoints — anpassen falls nicht via Skaffold port-forward
EMBEDDING_URL = "http://localhost:8080"
VECTORDB_DSN = "host=localhost port=5433 dbname=kamerplanter_vectors user=postgres password=devpassword"
OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "gemma3:4b"

# Eval-Daten
EVAL_DIR = Path("../../tests/rag-eval")
TOP_K = 5  # RAG chunks pro Frage

RAG_SYSTEM_PROMPT = (
    "Du bist ein erfahrener Pflanzenberater. Beantworte die Frage des Nutzers "
    "praezise und fachlich korrekt. Nutze den bereitgestellten Kontext aus der "
    "Wissensdatenbank. Wenn der Kontext nicht ausreicht, sage das ehrlich."
)

## 2. Verbindungstest

In [ ]:
# Embedding Service
try:
    r = httpx.get(f"{EMBEDDING_URL}/ready", timeout=5)
    print(f"Embedding Service: {r.json()}")
except Exception as e:
    print(f"Embedding Service: NICHT ERREICHBAR — {e}")

# VectorDB
try:
    with psycopg.connect(VECTORDB_DSN) as conn:
        count = conn.execute("SELECT COUNT(*) FROM ai_vector_chunks").fetchone()[0]
    print(f"VectorDB: {count} Chunks indexiert")
except Exception as e:
    print(f"VectorDB: NICHT ERREICHBAR — {e}")

# Ollama
try:
    r = httpx.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"Ollama: {len(models)} Modelle geladen — {models}")
    if OLLAMA_MODEL not in models and not any(OLLAMA_MODEL in m for m in models):
        print(f"  WARNUNG: {OLLAMA_MODEL} nicht geladen! Bitte erst: ollama pull {OLLAMA_MODEL}")
except Exception as e:
    print(f"Ollama: NICHT ERREICHBAR — {e}")

## 3. Service-Clients

In [ ]:
def embed_text(text: str) -> list[float]:
    """Embedding via externem Service."""
    resp = httpx.post(
        f"{EMBEDDING_URL}/embed",
        json={"texts": [text], "model": "all-MiniLM-L6-v2"},
        timeout=60.0,
    )
    resp.raise_for_status()
    return resp.json()["embeddings"][0]


def search_chunks(embedding: list[float], top_k: int = TOP_K) -> list[dict]:
    """Cosine-Similarity-Suche auf pgvector."""
    emb_str = f"[{','.join(str(v) for v in embedding)}]"
    sql = """
        SELECT source_key, source_type, title, content, metadata,
               1 - (embedding <=> %s::vector) AS score
        FROM ai_vector_chunks
        ORDER BY embedding <=> %s::vector
        LIMIT %s
    """
    with psycopg.connect(VECTORDB_DSN) as conn:
        rows = conn.execute(sql, (emb_str, emb_str, top_k)).fetchall()
    return [{"source_key": r[0], "title": r[2], "content": r[3], "score": float(r[5])} for r in rows]


def llm_generate(system_prompt: str, user_message: str) -> str:
    """Ollama Chat-Completion."""
    resp = httpx.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": OLLAMA_MODEL,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message},
            ],
            "stream": False,
            "options": {"num_predict": 1024, "temperature": 0.1},
        },
        timeout=120.0,
    )
    resp.raise_for_status()
    return resp.json().get("message", {}).get("content", "")

## 4. Eval-Daten laden

In [ ]:
with open(EVAL_DIR / "benchmark_questions.yaml", encoding="utf-8") as f:
    bench_data = yaml.safe_load(f)

with open(EVAL_DIR / "topic_synonyms.yaml", encoding="utf-8") as f:
    synonyms = yaml.safe_load(f).get("topics", {})

questions = bench_data.get("questions", [])
min_pass_score = bench_data.get("min_pass_score", 0.70)

# Uebersicht
categories = {}
for q in questions:
    cat = q.get("category", "unknown")
    categories[cat] = categories.get(cat, 0) + 1

print(f"{len(questions)} Benchmark-Fragen in {len(categories)} Kategorien:")
print(f"Pass-Schwelle: {min_pass_score:.0%}\n")
for cat, count in sorted(categories.items()):
    print(f"  {cat:<25s} {count:3d} Fragen")
print(f"\n{len(synonyms)} Topic-Synonyme geladen")

## 5. Benchmark ausfuehren

Jede Frage durchlaeuft die volle RAG-Pipeline:
1. Frage → Embedding Service → Query-Vektor
2. Query-Vektor → VectorDB → Top-K Chunks
3. Chunks + Frage → Ollama → Antwort
4. Antwort → Topic-Match → Score

In [ ]:
def topic_matches(topic_key: str, answer: str) -> bool:
    """Prueft ob ein Topic in der Antwort vorkommt."""
    topic_def = synonyms.get(topic_key, {})
    pattern = topic_def.get("pattern")
    if pattern and re.search(pattern, answer):
        return True
    keywords = topic_def.get("de", [])
    answer_lower = answer.lower()
    return any(kw.lower() in answer_lower for kw in keywords)


def evaluate_question(question: dict) -> dict:
    """Eine Frage durch die RAG-Pipeline evaluieren."""
    q_id = question.get("id", "unknown")
    q_text = question.get("question", "")
    expected_topics = question.get("expected_topics", [])
    expected_not = question.get("expected_NOT", [])

    # Kontext aufbauen
    ctx = question.get("context", {}) or {}
    context_parts = [f"{k}: {v}" for k, v in ctx.items() if v is not None]
    context_str = ", ".join(context_parts)

    # 1. Retrieve
    query = f"{q_text} {context_str}".strip()
    embedding = embed_text(query)
    chunks = search_chunks(embedding)

    # 2. Generate
    chunk_texts = "\n\n---\n\n".join(
        f"[{c['source_key']}] {c['title']}\n{c['content']}" for c in chunks
    )
    user_msg = f"Kontext aus Wissensdatenbank:\n{chunk_texts}\n\n"
    if context_str:
        user_msg += f"Situation: {context_str}\n\n"
    user_msg += f"Frage: {q_text}"

    try:
        answer = llm_generate(RAG_SYSTEM_PROMPT, user_msg)
    except Exception as exc:
        print(f"  LLM-Fehler bei {q_id}: {exc}")
        answer = ""

    # 3. Score
    hits = [t for t in expected_topics if topic_matches(t, answer)]
    misses = [t for t in expected_topics if not topic_matches(t, answer)]
    fps = [t for t in expected_not if topic_matches(t, answer)]

    total = len(expected_topics) if expected_topics else 1
    score = max(0.0, min(1.0, (len(hits) / total) - (len(fps) * 0.5)))

    return {
        "id": q_id,
        "category": question.get("category", "unknown"),
        "difficulty": question.get("difficulty", "unknown"),
        "score": score,
        "topic_hits": hits,
        "topic_misses": misses,
        "false_positives": fps,
        "answer": answer[:500],
        "chunks_used": len(chunks),
        "chunk_scores": [round(c["score"], 3) for c in chunks],
    }

In [ ]:
# Optionaler Filter — None = alle Fragen, oder z.B. ["diagnostik", "duengung"]
CATEGORY_FILTER = None

filtered = questions
if CATEGORY_FILTER:
    filtered = [q for q in questions if q.get("category") in CATEGORY_FILTER]

print(f"Evaluiere {len(filtered)} Fragen mit {OLLAMA_MODEL} ...\n")

results = []
for i, q in enumerate(filtered, 1):
    r = evaluate_question(q)
    status = "PASS" if r["score"] >= min_pass_score else "FAIL"
    print(f"  [{i:3d}/{len(filtered)}] {r['id']:<20s} {r['score']:.2f}  {status}", end="")
    if r["topic_misses"]:
        print(f"  misses: {', '.join(r['topic_misses'])}", end="")
    if r["false_positives"]:
        print(f"  FPs: {', '.join(r['false_positives'])}", end="")
    print()
    results.append(r)

print(f"\nFertig! {len(results)} Fragen evaluiert.")

## 6. Ergebnis-Tabelle

In [ ]:
df = pd.DataFrame(results)
df["passed"] = df["score"] >= min_pass_score

total_score = df["score"].mean()
total_passed = total_score >= min_pass_score

print(f"{'=' * 60}")
print(f"  Gesamt-Score:  {total_score:.2%}  {'PASS' if total_passed else 'FAIL'}")
print(f"  Schwelle:      {min_pass_score:.0%}")
print(f"  Fragen:        {len(df)}")
print(f"  Bestanden:     {df['passed'].sum()}/{len(df)}")
print(f"{'=' * 60}")

df[["id", "category", "difficulty", "score", "passed", "topic_misses", "false_positives"]]

## 7. Scores pro Kategorie

In [ ]:
cat_scores = df.groupby("category")["score"].agg(["mean", "min", "max", "count"])
cat_scores.columns = ["Score (Mittel)", "Min", "Max", "Fragen"]
cat_scores = cat_scores.sort_values("Score (Mittel)", ascending=False)
cat_scores["Status"] = cat_scores["Score (Mittel)"].apply(lambda s: "PASS" if s >= min_pass_score else "FAIL")
cat_scores

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#4caf50" if s >= min_pass_score else "#f44336" for s in cat_scores["Score (Mittel)"]]
bars = ax.barh(cat_scores.index, cat_scores["Score (Mittel)"], color=colors)
ax.axvline(x=min_pass_score, color="#ff9800", linestyle="--", linewidth=2, label=f"Schwelle ({min_pass_score:.0%})")
ax.set_xlim(0, 1.05)
ax.set_xlabel("Score")
ax.set_title(f"RAG Benchmark — Score pro Kategorie (Gesamt: {total_score:.1%})")
ax.legend()

for bar, score in zip(bars, cat_scores["Score (Mittel)"]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{score:.0%}", va="center", fontsize=10)

plt.tight_layout()
plt.show()

## 8. Score pro Schwierigkeitsgrad

In [ ]:
diff_scores = df.groupby("difficulty")["score"].agg(["mean", "count"])
diff_scores.columns = ["Score (Mittel)", "Fragen"]

order = ["beginner", "intermediate", "expert"]
diff_scores = diff_scores.reindex([d for d in order if d in diff_scores.index])

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#81c784", "#ffb74d", "#e57373"]
ax.bar(diff_scores.index, diff_scores["Score (Mittel)"], color=colors[:len(diff_scores)])
ax.axhline(y=min_pass_score, color="#ff9800", linestyle="--", linewidth=2)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("RAG Benchmark — Score pro Schwierigkeitsgrad")

for i, (idx, row) in enumerate(diff_scores.iterrows()):
    ax.text(i, row["Score (Mittel)"] + 0.02, f"{row['Score (Mittel)']:.0%}", ha="center")

plt.tight_layout()
plt.show()

## 9. Fehleranalyse — Schlechteste Fragen

In [ ]:
failures = df[~df["passed"]].sort_values("score")
print(f"{len(failures)} Fragen unter Schwelle ({min_pass_score:.0%}):\n")

for _, row in failures.head(15).iterrows():
    print(f"  {row['id']:<20s} Score: {row['score']:.2f}  Kat: {row['category']}")
    if row["topic_misses"]:
        print(f"    Fehlende Topics: {', '.join(row['topic_misses'])}")
    if row["false_positives"]:
        print(f"    Halluzinationen: {', '.join(row['false_positives'])}")
    print()

## 10. Einzelne Frage inspizieren

Aendere `INSPECT_ID` um eine bestimmte Frage im Detail zu sehen — inklusive RAG-Antwort und abgerufenen Chunks.

In [ ]:
INSPECT_ID = "diag-001"  # Hier aendern

row = df[df["id"] == INSPECT_ID]
if row.empty:
    print(f"Frage {INSPECT_ID} nicht gefunden")
else:
    r = row.iloc[0]
    q = next(q for q in questions if q.get("id") == INSPECT_ID)

    print(f"ID:         {r['id']}")
    print(f"Kategorie:  {r['category']}")
    print(f"Schwierig.: {r['difficulty']}")
    print(f"Score:      {r['score']:.2f}  {'PASS' if r['score'] >= min_pass_score else 'FAIL'}")
    print(f"\nFrage: {q['question']}")
    print(f"\nKontext: {q.get('context', {})}")
    print(f"\nErwartete Topics: {q.get('expected_topics', [])}")
    print(f"Hits:   {r['topic_hits']}")
    print(f"Misses: {r['topic_misses']}")
    print(f"FPs:    {r['false_positives']}")
    print(f"Chunks: {r['chunks_used']} (Scores: {r['chunk_scores']})")
    print(f"\n{'─' * 60}")
    print(f"RAG-Antwort:\n\n{r['answer']}")

## 11. Chunk-Retrieval-Qualitaet

Wie gut sind die abgerufenen Chunks? Hohe Cosine-Scores = gutes Retrieval.

In [ ]:
all_chunk_scores = [s for scores in df["chunk_scores"] for s in scores]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(all_chunk_scores, bins=30, color="#42a5f5", edgecolor="white")
ax.axvline(x=0.5, color="#f44336", linestyle="--", label="Minimum sinnvoll (0.5)")
ax.set_xlabel("Cosine Similarity")
ax.set_ylabel("Anzahl Chunks")
ax.set_title(f"Verteilung der Chunk-Similarity-Scores (n={len(all_chunk_scores)})")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Median: {sorted(all_chunk_scores)[len(all_chunk_scores) // 2]:.3f}")
print(f"Mean:   {sum(all_chunk_scores) / len(all_chunk_scores):.3f}")
print(f"Min:    {min(all_chunk_scores):.3f}")
print(f"Max:    {max(all_chunk_scores):.3f}")

## 12. Ergebnisse speichern

In [ ]:
output = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "model": OLLAMA_MODEL,
    "method": "topic_match",
    "total_score": round(total_score, 4),
    "pass": bool(total_passed),
    "min_pass_score": min_pass_score,
    "category_scores": {cat: round(row["Score (Mittel)"], 4) for cat, row in cat_scores.iterrows()},
    "questions_evaluated": len(df),
    "failures": [
        {"id": r["id"], "score": round(r["score"], 2), "misses": r["topic_misses"], "fps": r["false_positives"]}
        for _, r in failures.iterrows()
    ],
}

output_path = EVAL_DIR / "eval_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Ergebnisse gespeichert: {output_path}")